In [ ]:
!pip install -q transformers datasets accelerate scikit-learn pandas torch

In [ ]:
import os
import pandas as pd
import torch
from datasets import Dataset
from torch.utils.data import Dataset as TorchDataset
from sklearn.model_selection import train_test_split
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

In [ ]:
MODEL_NAME = 'google-t5/t5-base'
DATASET_PATH = '/content/C4200M_50k.csv'
OUTPUT_DIR = './results_t5_base'
FINAL_MODEL_DIR = './t5_base_grammar_model'

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU Model: Tesla T4


In [ ]:
print(f"Loading tokenizer and model for {MODEL_NAME}...")
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)

Loading tokenizer and model for google-t5/t5-base...


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  892MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
print(f"Loading dataset from {DATASET_PATH}...")
df = pd.read_csv(DATASET_PATH)

# 1. First split: 80% Train, 20% Temp (Validation + Test)
train_df, temp_df = train_test_split(df, test_size=0.20, random_state=42)

# 2. Second split: Divide the 20% equally into 10% Validation and 10% Holdout Test
eval_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42)

# 3. Save the test holdout set separately for evaluation later
test_df.to_csv('/content/test_holdout.csv', index=False)

print(f"Train samples: {len(train_df)} | Validation samples: {len(eval_df)} | Holdout Test samples: {len(test_df)}")

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(eval_df)

Loading dataset from /content/C4200M_50k.csv...
Train samples: 40000 | Validation samples: 5000 | Holdout Test samples: 5000


In [ ]:
class GrammarDataset(TorchDataset):
    def __init__(self, dataset, tokenizer, max_len=64):
        self.dataset = dataset
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.dataset)

    def tokenize_data(self, example):
        input_, target_ = example['input'], example['label']

        tokenized_inputs = self.tokenizer(
            input_,
            padding='max_length',
            max_length=self.max_len,
            return_attention_mask=True,
            truncation=True
        )
        tokenized_targets = self.tokenizer(
            target_,
            padding='max_length',
            max_length=self.max_len,
            return_attention_mask=True,
            truncation=True
        )

        return {
            "input_ids": tokenized_inputs['input_ids'],
            "attention_mask": tokenized_inputs['attention_mask'],
            "labels": tokenized_targets['input_ids']
        }

    def __getitem__(self, index):
        return self.tokenize_data(self.dataset[index])

In [ ]:
datacollator = DataCollatorForSeq2Seq(tokenizer, model=model)

args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,      # Batch size optimized for Colab T4 GPU
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,     # Effective batch size = 16 (8 * 2)
    fp16=True,                         # Mixed precision training
    optim="adamw_torch",
    num_train_epochs=3,
    logging_steps=500,
    save_total_limit=1,                # Keep only the latest checkpoint to save disk space
    load_best_model_at_end=True
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    processing_class=tokenizer,        # Changed from tokenizer=tokenizer
    train_dataset=GrammarDataset(train_dataset, tokenizer),
    eval_dataset=GrammarDataset(test_dataset, tokenizer),
    data_collator=datacollator,
)

In [ ]:
print("Starting training process...")
trainer.train()

print(f"Saving final fine-tuned model and tokenizer to {FINAL_MODEL_DIR}...")
model.save_pretrained(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)
print("Training complete!")

Starting training process...


Epoch,Training Loss,Validation Loss
1,0.642327,0.283136
2,0.592180,0.275946
3,0.582932,0.274339


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


Saving final fine-tuned model and tokenizer to ./t5_base_grammar_model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete!


In [ ]:
# Cell 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Cell 2: Copy local model folder to Drive
import shutil

# Make sure this matches the variable FINAL_MODEL_DIR from your code
source_dir = FINAL_MODEL_DIR  # e.g., "./t5_base_grammar_model"
destination_dir = '/content/drive/MyDrive/t5_base_grammar_model'

shutil.copytree(source_dir, destination_dir, dirs_exist_ok=True)
print(f"Model and tokenizer successfully saved to Google Drive at: {destination_dir}")

Model and tokenizer successfully saved to Google Drive at: /content/drive/MyDrive/t5_base_grammar_model
